<a href="https://colab.research.google.com/github/Nosaphi/LLM-Internship/blob/main/PipelineT5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets evaluate accelerate sentencepiece scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [2]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import torch
from datasets import Dataset, DatasetDict
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
import evaluate


In [5]:
# Loading datas line per line
raw_data = []
with open('/content/data.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            raw_data.append(json.loads(line))

In [6]:
def extract_top_category(label: str) -> str:
    """
    Extract top category from encoding
    """
    return str(label).split('.')[0].strip()

# Apply exrtraction
for item in raw_data:
    item['target'] = extract_top_category(item['label'])

df = pd.DataFrame(raw_data)


In [7]:
try:
    train_df, test_df = train_test_split(
        df,
        test_size=0.2,
        random_state=42,
        stratify=df['target']
    )

except ValueError as e:

    train_df, test_df = train_test_split(
        df,
        test_size=0.2,
        random_state=42
    )


In [8]:
PREFIX = "classify encoding: "

def df_to_hf_dataset(dataframe: pd.DataFrame) -> Dataset:
    """Convert Dataframe to Dataset."""
    return Dataset.from_dict({
        'input_text': [PREFIX + text for text in dataframe['text'].tolist()],
        'target_text': dataframe['target'].tolist(),
        'original_label': dataframe['label'].tolist(),
    })

dataset = DatasetDict({
    'train': df_to_hf_dataset(train_df),
    'test':  df_to_hf_dataset(test_df),
})

print(dataset)
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text', 'original_label'],
        num_rows: 69
    })
    test: Dataset({
        features: ['input_text', 'target_text', 'original_label'],
        num_rows: 18
    })
})
{'input_text': 'classify encoding: ed ADDRESS361  % ADDRESS1676 ralmone ADDRESS343) colinanziato ADDRESS252 } ; #: ADDRESS96 PERSON508 ie  fuedi ADDRESS1979 plus enza) dall\'unione europea  informativa effettuata ai sensi dell\'art, 13 regolamento ( ue) PHONE1743 (rgdp)  nf.  . .  " api ADDRESS1832 = iloi dara : \' ADDRESS569  i ADDRESS2 di ADDRESS9 ADDRESS473PERSON290. con ADDRESS155 in ADDRESS9 ADDRESS473PERSON290. ADDRESS185 eligio PERSON412 ADDRESS343.ADDRESS131.ADDRESS1435.. pece:  protocollo\' ADDRESS125.ADDRESS2.quartusantelena.ADDRESS4.ADDRESS250. PERSON691: PHONE2192. nella PERSON333 qualità di ente capolila del \' " \' ; ADDRESS206 ADDRESS1681 ali cpi { "2  plus ADDRESS9 parteolla. ADDRESS238 del trattamento dei dati. tratterà i dat

In [9]:
MODEL_NAME = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

# Setting
MAX_INPUT_LENGTH  = 256
MAX_TARGET_LENGTH = 8

def preprocess_function(examples):
    """Tokenize T5's entries and output."""
    model_inputs = tokenizer(
        examples['input_text'],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
    )

    # Tokenize the labels
    labels = tokenizer(
        examples['target_text'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=['input_text', 'target_text', 'original_label'],
)

print(tokenized_dataset)
print(tokenized_dataset['train'][0])

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 69
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 18
    })
})
{'input_ids': [853, 4921, 3, 35, 9886, 10, 3, 15, 26, 8502, 3913, 10087, 3420, 536, 3, 1454, 8502, 3913, 10087, 2938, 3959, 3, 4900, 2157, 15, 8502, 3913, 10087, 3710, 5268, 7632, 77, 4557, 23, 9, 235, 8502, 3913, 10087, 1828, 357, 3, 2, 3, 117, 1713, 10, 8502, 3913, 10087, 4314, 3, 8742, 10466, 1752, 927, 3, 23, 15, 7683, 15, 26, 23, 8502, 3913, 10087, 2294, 4440, 303, 3, 35, 1629, 61, 836, 195, 31, 16598, 15, 3, 28188, 9, 3261, 4572, 9, 4154, 17, 76, 144, 9, 3, 9, 23, 3952, 23, 20, 195, 31, 1408, 6, 1179, 3, 60, 7579, 9, 297, 32, 41, 3, 76, 15, 61, 3, 8023, 7894, 2517, 4906, 41, 52, 122, 26, 102, 61, 3, 29, 89, 5, 3, 5, 3, 5, 96, 3, 13306, 8502, 3913, 10087, 2606, 2668, 3274, 3, 173, 32, 23, 649, 9, 3, 10, 3, 31, 8502, 3913, 10087, 755, 3951, 3, 23, 85

In [11]:
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

# Number of settings
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Paramètres totaux    : {total_params:,}")
print(f"Paramètres entraînables : {trainable_params:,}")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Paramètres totaux    : 60,506,624
Paramètres entraînables : 60,506,624


In [10]:
def compute_metrics(eval_preds):
    """Calcule l'accuracy exacte entre prédictions et labels."""
    predictions, labels = eval_preds

    # Decode the predictions
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Replace -100 by the padding's token
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Cleaning
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    correct = sum(p == l for p, l in zip(decoded_preds, decoded_labels))
    accuracy = correct / len(decoded_labels)

    return {
        'accuracy': round(accuracy, 4),
        'correct': correct,
        'total': len(decoded_labels),
    }

In [12]:
OUTPUT_DIR = "./t5_encoding_classifier"

# Training Hypersettings
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=40,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-4,
    warmup_steps=0.1,
    weight_decay=0.01,
    lr_scheduler_type="cosine",

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    predict_with_generate=True,
    generation_max_length=8,
    generation_num_beams=4,

    logging_dir="./logs",
    logging_steps=5,
    report_to="none",

    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    seed=42,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [13]:
# Data collator with dynamic padding
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,  # ignored in loss calcul
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)


In [14]:
train_result = trainer.train()

print("\n=== Résultats d'entraînement ===")
print(f"Loss finale       : {train_result.training_loss:.4f}")
print(f"Steps totaux      : {train_result.global_step}")
print(f"Durée             : {train_result.metrics['train_runtime']:.1f}s")

# Save best model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Epoch,Training Loss,Validation Loss,Accuracy,Correct,Total
1,5.465020,4.706351,0.000000,0,18
2,2.777331,2.637742,0.055600,1,18
3,1.058278,1.662608,0.388900,7,18
4,0.537143,1.500318,0.222200,4,18
5,0.267094,1.429840,0.388900,7,18
6,0.304689,1.613455,0.333300,6,18
7,0.210556,1.519945,0.444400,8,18
8,0.140307,1.378986,0.500000,9,18
9,0.064894,1.508563,0.333300,6,18
10,0.030038,1.698804,0.333300,6,18


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].



=== Résultats d'entraînement ===
Loss finale       : 0.9335
Steps totaux      : 234
Durée             : 134.1s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./t5_encoding_classifier/tokenizer_config.json',
 './t5_encoding_classifier/tokenizer.json')

In [15]:
eval_results = trainer.evaluate()

print("\n=== Résultats sur le jeu de test ===")
for k, v in eval_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")


=== Résultats sur le jeu de test ===
  eval_loss: 1.3790
  eval_accuracy: 0.5000
  eval_correct: 9
  eval_total: 18
  eval_runtime: 1.9882
  eval_samples_per_second: 9.0540
  eval_steps_per_second: 2.5150
  epoch: 13.0000


In [20]:
# Create predicts on test
predictions_output = trainer.predict(tokenized_dataset['test'])

raw_preds  = predictions_output.predictions
raw_labels = predictions_output.label_ids

raw_labels = np.where(raw_labels != -100, raw_labels, tokenizer.pad_token_id)

decoded_preds  = [p.strip() for p in tokenizer.batch_decode(raw_preds,  skip_special_tokens=True)]
decoded_labels = [l.strip() for l in tokenizer.batch_decode(raw_labels, skip_special_tokens=True)]

# DataFrame de résultats
results_df = pd.DataFrame({
    'texte':            test_df['text'].values,
    'label_complet':    test_df['label'].values,
    'categorie_reelle': decoded_labels,
    'categorie_predite': decoded_preds,
    'correct':          [p == l for p, l in zip(decoded_preds, decoded_labels)]
})

print("=== Prédictions sur le jeu de test ===")
#print(results_df.to_string(index=False))

print(f"\n Accuracy : {results_df['correct'].mean():.1%}")
print(f"   ({results_df['correct'].sum()}/{len(results_df)} exemples corrects)")

=== Prédictions sur le jeu de test ===

 Accuracy : 50.0%
   (9/18 exemples corrects)


In [17]:
def predict_category(text: str, model=model, tokenizer=tokenizer, device=None) -> dict:
    """Prédit la catégorie principale pour un texte donné.

    Returns:
        dict avec 'predicted_category', 'confidence_tokens'
    """
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'

    model.eval()
    model.to(device)

    input_text = PREFIX + text
    inputs = tokenizer(
        input_text,
        return_tensors='pt',
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=8,
            num_beams=4,
            early_stopping=True,
        )

    predicted = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    return {
        'text': text,
        'predicted_category': predicted,
    }


In [18]:
# Save best model and tokenizer
FINAL_MODEL_DIR = "./t5_encoding_classifier_final"
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

# Save metadata of the model
categories = sorted(df['target'].unique())
metadata = {
    'base_model': MODEL_NAME,
    'categories': categories,
    'n_categories': len(categories),
    'train_size': len(train_df),
    'test_size': len(test_df),
    'test_accuracy': results_df['correct'].mean(),
    'prefix': PREFIX,
    'max_input_length': MAX_INPUT_LENGTH,
}

with open(f"{FINAL_MODEL_DIR}/metadata.json", 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"Métadonnées : {metadata}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Métadonnées : {'base_model': 't5-small', 'categories': ['Ambiente: autorizzazioni, monitoraggio e controllo', 'Amministrazione generale', 'Attività ed eventi culturali', 'Opere pubbliche', 'PUBBLICA ISTRUZIONE', 'Pianificazione e gestione del territorio', 'Polizia locale e sicurezza pubblica', 'Risorse finanziarie e patrimonio', 'Servizi alla persona', 'Servizi demografici', 'Stato civile'], 'n_categories': 11, 'train_size': 69, 'test_size': 18, 'test_accuracy': np.float64(0.5), 'prefix': 'classify encoding: ', 'max_input_length': 256}


In [19]:
# Load saved model
from transformers import T5ForConditionalGeneration, T5Tokenizer

loaded_model     = T5ForConditionalGeneration.from_pretrained(FINAL_MODEL_DIR)
loaded_tokenizer = T5Tokenizer.from_pretrained(FINAL_MODEL_DIR)


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]